# Aprendizaje automático interpretable para estratificar el riesgo de eventos adversos materno-perinatales

## Cuaderno reproducible para Google Colab — versión (estable)

**Estudio:** Aprendizaje automático interpretable para estratificar el riesgo de eventos adversos materno-perinatales en La Guajira (Colombia), 2016–2022.

**Manuscrito:** AG-0.9-2026 — Revista Facultad Nacional de Salud Pública (Universidad de Antioquia).

---

### ⚠️ Nota importante sobre versiones (lee antes de ejecutar)

Esta versión del cuaderno está diseñada para funcionar **sin tocar las versiones de las librerías preinstaladas en Colab**.

**Si tu sesión está rota:**
1. Menú `Entorno de ejecución` → `Restablecer el entorno de ejecución a los valores predeterminados de fábrica` (no solo "Reiniciar"; **Restablecer de fábrica**).
2. Vuelve a ejecutar desde el Paso 1 de este cuaderno.

---

### Cómo se usa este cuaderno

1. Subir el archivo `dataset_MME_LaGuajira_2016_2022.xlsx` cuando el cuaderno lo solicite (Paso 4). Acepta tanto Google Drive como subida directa.
2. Menú `Entorno de ejecución` → `Ejecutar todas`. Tiempo total: 3 a 5 minutos.

### Marco causal del estudio *(véase Figura 1 del manuscrito)*

```
Determinantes distales  →  Mediadores estructurales  →  Complicaciones clínicas  →  MME severa
  edad, etnia, ruralidad     acceso, CPN efectivo,         hipertensión, hemorragia,    (desenlace OMS)
                              pertinencia intercultural     sepsis
```

> ⚠️ **Nota ética:** Los datos están anonimizados (Resolución 8430 de 1993; Ley 1581 de 2012). Aval del Comité Institucional de Ética en Investigación, Universidad de La Guajira, acta n.° 04 del 12 de marzo de 2023.


## Paso 1. Instalación de dependencias

Colab trae preinstalados `numpy`, `pandas`, `scipy`, `statsmodels`, `scikit-learn`, `matplotlib`, `seaborn`, `xgboost` y `openpyxl` en versiones compatibles entre sí. La única librería que **no** trae por defecto y que necesitamos es `imbalanced-learn` (para SMOTE).

**No fijamos versiones específicas a propósito**: hacerlo rompería el equilibrio del entorno de Colab. Si necesitas reproducir un experimento con versiones idénticas, ejecuta el último bloque del Paso 1 que imprime el listado completo de versiones encontradas.

In [ ]:
# Instalamos UNA SOLA librería: imbalanced-learn (la única que Colab no incluye)
# La opción -q oculta mensajes verbosos; no fijamos versión para evitar conflictos.
!pip install -q imbalanced-learn

print("✓ imbalanced-learn instalado.\n")


In [ ]:
# Verificación de versiones (informativo; no falla si alguna es distinta)
import sys, importlib

libs = ['numpy','pandas','scipy','statsmodels','sklearn','matplotlib',
        'seaborn','xgboost','imblearn','openpyxl']

print(f"Python: {sys.version.split()[0]}")
print("-" * 50)
for lib in libs:
    try:
        m = importlib.import_module(lib)
        ver = getattr(m, '__version__', 'desconocida')
        print(f"  {lib:18s} {ver}")
    except ImportError as e:
        print(f"  {lib:18s} ✗ NO INSTALADO ({e})")
print("-" * 50)
print("Si todas aparecen con versión, el ambiente está listo.")


## Paso 2. Importación de librerías

Agrupamos las importaciones por funcionalidad. Si alguna falla con `ImportError`, asegúrate de haber ejecutado el Paso 1 completo.

In [ ]:
# === Manipulación de datos ===
import numpy as np
import pandas as pd
import warnings; warnings.filterwarnings('ignore')

# === Visualización ===
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# === Machine Learning (scikit-learn) ===
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              precision_score, recall_score, balanced_accuracy_score,
                              brier_score_loss, roc_curve, precision_recall_curve,
                              confusion_matrix, classification_report)

# === XGBoost ===
from xgboost import XGBClassifier

# === Manejo de desbalance ===
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline  # usar el Pipeline de imblearn,
                                                         # NO el de sklearn, para que SMOTE
                                                         # se aplique solo en entrenamiento.

# === Estadística clásica ===
import statsmodels.api as sm
from scipy import stats

# === Carga de archivos en Colab ===
from google.colab import files, drive

print("✓ Todas las librerías importadas correctamente.")


## Paso 3. Configuración visual

Parámetros gráficos consistentes con el manuscrito: tipografía serif (compatible con Times New Roman), tamaños legibles, paleta **Okabe-Ito** segura para daltonismo y legible también en escala de grises.

In [ ]:
# === Tipografía ===
mpl.rcParams['font.family']     = 'serif'
mpl.rcParams['font.size']       = 12
mpl.rcParams['axes.titlesize']  = 13
mpl.rcParams['axes.labelsize']  = 12
mpl.rcParams['xtick.labelsize'] = 11
mpl.rcParams['ytick.labelsize'] = 11
mpl.rcParams['legend.fontsize'] = 10

# === Apariencia limpia ===
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top']   = False
mpl.rcParams['axes.linewidth']    = 0.9

# === Resolución alta para publicación ===
mpl.rcParams['savefig.dpi'] = 600
mpl.rcParams['figure.dpi']  = 100

# === Paleta Okabe-Ito (color-blind safe) ===
OK = {
    'azul':    '#0072B2',
    'naranja': '#E69F00',
    'verde':   '#009E73',
    'amarillo':'#F0E442',
    'celeste': '#56B4E9',
    'rojo':    '#D55E00',
    'rosa':    '#CC79A7',
    'gris':    '#5A5A5A',
    'negro':   '#000000',
}

# === Semilla para reproducibilidad exacta ===
SEMILLA = 42
np.random.seed(SEMILLA)

print(f"✓ Configuración visual aplicada (semilla = {SEMILLA}).")


## Paso 4. Carga del dataset

Esta versión soporta **dos métodos** de carga, en orden de preferencia:

1. **Google Drive** — si el archivo `dataset_MME_LaGuajira_2016_2022.xlsx` está en la raíz de tu Drive, se monta y se busca automáticamente.
2. **Subida directa** — si Drive no encuentra el archivo, se abre un diálogo para subirlo desde tu equipo.

El archivo debe tener tres hojas: `datos`, `diccionario_variables` y `metadata`.

In [ ]:
# === Intento 1: Google Drive ===
NOMBRE_ARCHIVO = 'dataset_MME_LaGuajira_2016_2022.xlsx'
ruta_archivo = None

import os
try:
    if not os.path.ismount('/content/drive'):
        print("Montando Google Drive (se abrirá un diálogo de autorización)...")
        drive.mount('/content/drive', force_remount=False)

    # Buscamos el archivo en las ubicaciones más comunes
    candidatos = [
        f'/content/drive/MyDrive/{NOMBRE_ARCHIVO}',
        f'/content/drive/My Drive/{NOMBRE_ARCHIVO}',
        f'/content/drive/MyDrive/Colab Notebooks/{NOMBRE_ARCHIVO}',
    ]
    for c in candidatos:
        if os.path.exists(c):
            ruta_archivo = c
            print(f"✓ Archivo encontrado en Drive: {ruta_archivo}")
            break

    if ruta_archivo is None:
        print("⚠️  Archivo no encontrado en Drive. Pasamos a subida directa.")
except Exception as e:
    print(f"⚠️  No se pudo usar Drive ({e}). Pasamos a subida directa.")

# === Intento 2: Subida directa ===
if ruta_archivo is None:
    print("\nSube el archivo dataset_MME_LaGuajira_2016_2022.xlsx desde tu equipo:")
    subido = files.upload()
    if subido:
        ruta_archivo = next(iter(subido.keys()))
        print(f"✓ Archivo subido: {ruta_archivo}")
    else:
        raise FileNotFoundError("No se subió ningún archivo. Vuelve a ejecutar esta celda.")


In [ ]:
# Leemos las tres hojas del Excel
df_datos       = pd.read_excel(ruta_archivo, sheet_name='datos')
df_diccionario = pd.read_excel(ruta_archivo, sheet_name='diccionario_variables')
df_metadata    = pd.read_excel(ruta_archivo, sheet_name='metadata')

print(f"Hoja 'datos':       {df_datos.shape[0]} filas × {df_datos.shape[1]} columnas")
print(f"Hoja 'diccionario': {df_diccionario.shape[0]} variables documentadas")
print(f"Hoja 'metadata':    {df_metadata.shape[0]} campos del estudio")

print("\n=== Primeras 5 filas ===")
df_datos.head()


In [ ]:
# Diccionario de variables
df_diccionario


In [ ]:
# Metadata del estudio
df_metadata


## Paso 5. Análisis exploratorio

Estructura, faltantes, distribución del desenlace y descriptivas continuas. Cumple los ítems 13–14 de RECORD.

In [ ]:
print("=== ESTRUCTURA ===")
df_datos.info()


In [ ]:
print("=== VALORES FALTANTES ===")
missing = df_datos.isna().sum()
missing_pct = (missing / len(df_datos) * 100).round(2)
pd.DataFrame({'faltantes_n': missing, 'faltantes_pct': missing_pct})


In [ ]:
print("=== DISTRIBUCIÓN DE MME SEVERA ===")
conteo = df_datos['MME_severa'].value_counts().sort_index()
prev = df_datos['MME_severa'].mean() * 100
print(f"  Casos negativos (0): {conteo.get(0, 0):>5}")
print(f"  Casos positivos (1): {conteo.get(1, 0):>5}")
print(f"  Prevalencia:         {prev:.2f} %")
print(f"\n  ⚠️  Desbalance marcado (< 5 %) → justifica SMOTE y AUPRC.")


In [ ]:
print("=== DESCRIPTIVAS CONTINUAS ===")
continuas = ['edad_materna','numero_gestaciones','antecedente_abortos',
             'numero_controles_prenatales','semana_inicio_control_prenatal']
df_datos[continuas].describe().round(2)


## Paso 6. Preprocesamiento

- Imputación con mediana en faltantes (< 5 %).
- Estandarización dentro del pipeline (no aquí, para evitar filtración).
- Separación X (predictores) / y (desenlace).

In [ ]:
predictores = [c for c in df_datos.columns if c != 'MME_severa']
X = df_datos[predictores].copy()
y = df_datos['MME_severa'].astype(int)

for col in X.columns:
    if X[col].isna().any():
        X[col] = X[col].fillna(X[col].median())

print(f"X: {X.shape[0]} filas × {X.shape[1]} columnas")
print(f"y: {y.shape[0]} obs, {y.sum()} casos positivos ({y.mean()*100:.2f} %)")
print(f"✓ Sin faltantes: {X.isna().sum().sum() == 0}")


## Paso 7. Análisis bivariado

U de Mann-Whitney para continuas; chi-cuadrado de Pearson para categóricas.

In [ ]:
def biv_cont(df, var, t='MME_severa'):
    g0 = df[df[t]==0][var].dropna(); g1 = df[df[t]==1][var].dropna()
    _, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
    return {'variable': var, 'mediana_sin': g0.median(),
            'mediana_con': g1.median(), 'p_valor': p}

def biv_cat(df, var, t='MME_severa'):
    ct = pd.crosstab(df[var], df[t])
    _, p, _, _ = stats.chi2_contingency(ct)
    return {'variable': var, 'mediana_sin': '—',
            'mediana_con': '—', 'p_valor': p}

filas = []
for v in ['edad_materna','numero_gestaciones','antecedente_abortos',
         'numero_controles_prenatales','semana_inicio_control_prenatal']:
    filas.append(biv_cont(df_datos, v))
for v in ['area_residencia','pertenencia_etnica','grupo_indigena']:
    filas.append(biv_cat(df_datos, v))

tab_biv = pd.DataFrame(filas)
tab_biv['significativo'] = tab_biv['p_valor'] < 0.05
tab_biv.round(4)


## Paso 8. Regresión logística multivariada (Figura 5 del artículo)

OR ajustadas con IC 95 % sobre variables categorizadas según el DAG.

In [ ]:
df_log = df_datos.copy()
df_log['edad_<20']           = (df_log['edad_materna']                  < 20).astype(int)
df_log['edad_>=35']          = (df_log['edad_materna']                 >= 35).astype(int)
df_log['rural']              = (df_log['area_residencia']               == 3).astype(int)
df_log['indigena']           = (df_log['pertenencia_etnica']            == 1).astype(int)
df_log['CPN_insuficiente']   = (df_log['numero_controles_prenatales']   <  4).astype(int)
df_log['inicio_CPN_tardio']  = (df_log['semana_inicio_control_prenatal']> 16).astype(int)
df_log['multiparidad']       = (df_log['numero_gestaciones']           >= 4).astype(int)

cols_log = ['edad_<20','edad_>=35','rural','indigena',
            'CPN_insuficiente','inicio_CPN_tardio','multiparidad']
X_log = df_log[cols_log].astype(float)
y_log = df_log['MME_severa']

X_log_sm = sm.add_constant(X_log)
modelo = sm.Logit(y_log, X_log_sm).fit(disp=0, maxiter=200)

params = modelo.params; conf = modelo.conf_int(); ps = modelo.pvalues
OR = np.exp(params); ORlo = np.exp(conf[0]); ORhi = np.exp(conf[1])
tabla_OR = pd.DataFrame({
    'Variable': params.index, 'OR': OR.values.round(3),
    'IC95_inf': ORlo.values.round(3), 'IC95_sup': ORhi.values.round(3),
    'p_valor': ps.values.round(3),
})
tabla_OR['significativo'] = tabla_OR['p_valor'] < 0.05
print(f"N = {len(df_log)}, eventos = {int(y_log.sum())}, prev = {y_log.mean()*100:.2f} %")
print(f"Pseudo R²: {modelo.prsquared:.4f}\n")
tabla_OR


## Paso 9. Forest plot (Figura 5)

In [ ]:
reg = tabla_OR[tabla_OR['Variable']!='const'].copy()
etiquetas = {
    'edad_<20':'Edad materna < 20 años','edad_>=35':'Edad materna ≥ 35 años',
    'rural':'Residencia rural dispersa','indigena':'Pertenencia étnica indígena',
    'CPN_insuficiente':'Controles prenatales < 4','inicio_CPN_tardio':'Inicio CPN > 16 semanas',
    'multiparidad':'Multiparidad ≥ 4 gestaciones',
}
reg['Etiqueta'] = reg['Variable'].map(etiquetas)
reg = reg.sort_values('OR')

fig, ax = plt.subplots(figsize=(11, 5.5))
y_pos = np.arange(len(reg))
for i, fila in enumerate(reg.itertuples()):
    if fila.p_valor < 0.05 and fila.IC95_inf > 1:
        c, mk, ls = OK['rojo'], 'D', '-'
    elif fila.p_valor < 0.05 and fila.IC95_sup < 1:
        c, mk, ls = OK['azul'], 's', '-'
    else:
        c, mk, ls = OK['gris'], 'o', '--'
    ax.plot([fila.IC95_inf, fila.IC95_sup], [i, i], color=c, linewidth=2.5, linestyle=ls, alpha=0.85)
    ax.scatter([fila.OR], [i], s=140, color=c, zorder=10, edgecolor='black', linewidth=1.1, marker=mk)
    sig = ' *' if fila.p_valor < 0.05 else ''
    ax.text(fila.IC95_sup*1.04, i,
            f'OR = {fila.OR:.2f} [{fila.IC95_inf:.2f}–{fila.IC95_sup:.2f}]; p = {fila.p_valor:.3f}{sig}',
            va='center', fontsize=10, color='#222')

ax.axvline(1, color='black', linestyle=':', linewidth=1.4, alpha=0.85)
ax.text(0.45, len(reg)+0.4, 'Protector',       ha='center', fontsize=10, color=OK['azul'], fontweight='bold')
ax.text(2.0,  len(reg)+0.4, 'Factor de riesgo', ha='center', fontsize=10, color=OK['rojo'], fontweight='bold')

from matplotlib.lines import Line2D
leg = [
    Line2D([0],[0], marker='D', color='w', label='Factor de riesgo (p < 0,05)',
           markerfacecolor=OK['rojo'], markersize=10, markeredgecolor='black'),
    Line2D([0],[0], marker='s', color='w', label='Protector (p < 0,05)',
           markerfacecolor=OK['azul'], markersize=10, markeredgecolor='black'),
    Line2D([0],[0], marker='o', color='w', label='No significativo',
           markerfacecolor=OK['gris'], markersize=10, markeredgecolor='black'),
]
ax.legend(handles=leg, loc='lower right', fontsize=9.5, frameon=True, edgecolor='#999')
ax.set_yticks(y_pos); ax.set_yticklabels(reg['Etiqueta'], fontsize=11)
ax.set_xscale('log'); ax.set_xlim(0.3, 5.5)
ax.set_xlabel('Razón de odds ajustada (IC 95 %)  —  escala logarítmica', fontsize=11.5)
ax.grid(axis='x', alpha=0.25, linestyle=':', which='both')
ax.set_axisbelow(True); ax.set_ylim(-0.7, len(reg)+0.9)
plt.tight_layout()
plt.savefig('figura5_forest_OR.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ figura5_forest_OR.png")


## Paso 10. Comparación de ocho algoritmos (Figura 2)

In [ ]:
modelos = {
    'Reg. logística':    LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEMILLA),
    'Árbol decisión':    DecisionTreeClassifier(class_weight='balanced', random_state=SEMILLA, max_depth=6),
    'KNN (k=15)':        KNeighborsClassifier(n_neighbors=15),
    'Naive Bayes':       GaussianNB(),
    'SVM (RBF)':         SVC(probability=True, class_weight='balanced', random_state=SEMILLA),
    'Random Forest':     RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                                 random_state=SEMILLA, n_jobs=-1, max_depth=10),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=SEMILLA, max_depth=4),
    'XGBoost':           XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                        scale_pos_weight=(y==0).sum()/(y==1).sum(),
                                        random_state=SEMILLA, eval_metric='auc', verbosity=0),
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
resultados = []
print("Entrenando 8 modelos × 5 folds (1–3 minutos)...\n")
for nombre, base in modelos.items():
    pipe = ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote',  SMOTE(random_state=SEMILLA, k_neighbors=5)),
        ('clf',    base),
    ])
    aucs, aps, f1s, recs, precs, baccs, briers = [], [], [], [], [], [], []
    for tr, te in skf.split(X, y):
        Xt, Xv, yt, yv = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]
        pipe.fit(Xt, yt)
        proba = pipe.predict_proba(Xv)[:, 1]
        pred  = (proba >= 0.5).astype(int)
        aucs.append(roc_auc_score(yv, proba))
        aps.append(average_precision_score(yv, proba))
        f1s.append(f1_score(yv, pred, zero_division=0))
        recs.append(recall_score(yv, pred, zero_division=0))
        precs.append(precision_score(yv, pred, zero_division=0))
        baccs.append(balanced_accuracy_score(yv, pred))
        briers.append(brier_score_loss(yv, proba))
    resultados.append({
        'Modelo': nombre, 'AUC_media': np.mean(aucs), 'AUC_sd': np.std(aucs),
        'AUC_ic95_inf': np.mean(aucs) - 1.96*np.std(aucs)/np.sqrt(5),
        'AUC_ic95_sup': np.mean(aucs) + 1.96*np.std(aucs)/np.sqrt(5),
        'AUPRC': np.mean(aps), 'F1': np.mean(f1s), 'Sensibilidad': np.mean(recs),
        'Precisión': np.mean(precs), 'Exact_balanc': np.mean(baccs), 'Brier': np.mean(briers),
    })
    print(f"  {nombre:22s}  AUC = {np.mean(aucs):.3f} ± {np.std(aucs):.3f}  AUPRC = {np.mean(aps):.3f}")

tabla_modelos = pd.DataFrame(resultados).sort_values('AUC_media', ascending=False).reset_index(drop=True)
print("\n=== RESUMEN ===")
tabla_modelos.round(3)


## Paso 11. Figura 2: AUC con IC 95 %

In [ ]:
res_ord = tabla_modelos.sort_values('AUC_media', ascending=True).reset_index(drop=True)
maxauc = res_ord['AUC_media'].max()
top3_idx = res_ord['AUC_media'].nlargest(3).index.tolist()
fig, ax = plt.subplots(figsize=(9, 5.8))
y_pos = np.arange(len(res_ord))
colors = []
for i, auc in enumerate(res_ord['AUC_media']):
    if auc == maxauc: colors.append(OK['rojo'])
    elif i in top3_idx: colors.append(OK['naranja'])
    else: colors.append(OK['azul'])
ax.errorbar(res_ord['AUC_media'], y_pos,
            xerr=[res_ord['AUC_media']-res_ord['AUC_ic95_inf'],
                  res_ord['AUC_ic95_sup']-res_ord['AUC_media']],
            fmt='none', ecolor='#444', capsize=5, capthick=1.2, linewidth=1.2, zorder=2)
for i, (auc, c) in enumerate(zip(res_ord['AUC_media'], colors)):
    ax.scatter([auc], [i], s=180, color=c, zorder=10, edgecolor='black', linewidth=1.2, marker='o')
for i, auc in enumerate(res_ord['AUC_media']):
    ax.text(auc + 0.005, i + 0.18, f'{auc:.3f}', fontsize=9.5, fontweight='bold', color='#222')
ax.axvline(0.5, color='#888', linestyle=':',  linewidth=1.4, label='Azar (AUC = 0,50)')
ax.axvline(0.7, color=OK['verde'], linestyle='--', linewidth=1.4, label='Umbral aceptable (AUC = 0,70)')
best_i = res_ord['AUC_media'].idxmax()
ax.axhspan(best_i - 0.4, best_i + 0.4, color=OK['rojo'], alpha=0.08, zorder=0)
ax.set_yticks(y_pos); ax.set_yticklabels(res_ord['Modelo'], fontsize=11)
ax.set_xlabel('Área bajo la curva ROC (validación cruzada 5-fold, IC 95 %)', fontsize=12)
ax.set_xlim(0.55, 0.82)
ax.legend(loc='lower right', frameon=True, edgecolor='#888', fontsize=10)
ax.grid(axis='x', alpha=0.25, linestyle=':'); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('figura2_AUC_comparacion.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ figura2_AUC_comparacion.png")


## Paso 12. Importancia por permutación (Figura 3)

In [ ]:
pipe_rf = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=SEMILLA, k_neighbors=5)),
    ('clf',    RandomForestClassifier(n_estimators=500, class_weight='balanced',
                                       random_state=SEMILLA, n_jobs=-1, max_depth=10)),
])
pipe_rf.fit(X, y)
print("Calculando importancia por permutación (10 reps × 12 variables)...")
perm = permutation_importance(pipe_rf, X, y, n_repeats=10,
                              random_state=SEMILLA, n_jobs=-1, scoring='roc_auc')
imp_df = pd.DataFrame({
    'Variable': X.columns,
    'Importancia_media': perm.importances_mean,
    'Importancia_sd': perm.importances_std,
}).sort_values('Importancia_media', ascending=True)
imp_df.sort_values('Importancia_media', ascending=False).round(4)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
norm = imp_df['Importancia_media'] / imp_df['Importancia_media'].max()
colors_v = plt.cm.viridis(0.15 + 0.7 * norm)
ax.barh(imp_df['Variable'], imp_df['Importancia_media'],
        xerr=imp_df['Importancia_sd'],
        color=colors_v, edgecolor='black', linewidth=0.7,
        error_kw={'elinewidth': 1.2, 'capsize': 4, 'capthick': 1, 'ecolor':'#333'})
for i, (val, sd) in enumerate(zip(imp_df['Importancia_media'], imp_df['Importancia_sd'])):
    if val > 0.0001:
        ax.text(val + sd + 0.005, i, f'{val:.3f} ± {sd:.3f}',
                va='center', fontsize=9.5, color='#222', fontweight='bold')
ax.set_xlabel('Reducción de AUC al permutar la variable (± DE)', fontsize=11.5)
ax.axvline(0, color='black', linewidth=0.7)
ax.set_xlim(-0.005, imp_df['Importancia_media'].max() * 1.45)
ax.grid(axis='x', alpha=0.25, linestyle=':'); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('figura3_importancia.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ figura3_importancia.png")


## Paso 13. Curvas ROC y precisión–recall (Figura 4)

In [ ]:
modelos_top = {
    'Reg. logística':    LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEMILLA),
    'Random Forest':     RandomForestClassifier(n_estimators=500, class_weight='balanced',
                                                 random_state=SEMILLA, n_jobs=-1, max_depth=10),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=SEMILLA, max_depth=4),
    'XGBoost':           XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                        scale_pos_weight=(y==0).sum()/(y==1).sum(),
                                        random_state=SEMILLA, eval_metric='auc', verbosity=0),
}
all_y = {n:[] for n in modelos_top}
all_p = {n:[] for n in modelos_top}
for tr, te in skf.split(X, y):
    Xt, Xv, yt, yv = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]
    for nombre, base in modelos_top.items():
        pipe = ImbPipeline([
            ('s',  StandardScaler()),
            ('sm', SMOTE(random_state=SEMILLA, k_neighbors=5)),
            ('c',  base),
        ])
        pipe.fit(Xt, yt)
        proba = pipe.predict_proba(Xv)[:, 1]
        all_y[nombre].extend(yv.tolist())
        all_p[nombre].extend(proba.tolist())

estilos = [
    ('Reg. logística',    OK['azul'],    '--', 's'),
    ('Gradient Boosting', OK['verde'],   '-.', '^'),
    ('XGBoost',           OK['naranja'], ':',  'D'),
    ('Random Forest',     OK['rojo'],    '-',  'o'),
]
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for n, c, ls, mk in estilos:
    fpr, tpr, _ = roc_curve(all_y[n], all_p[n])
    a = roc_auc_score(all_y[n], all_p[n])
    axes[0].plot(fpr, tpr, color=c, linewidth=2.4, linestyle=ls,
                  label=f"{n} (AUC = {a:.3f})", alpha=0.95)
axes[0].plot([0,1],[0,1], color='#666', linestyle=':', linewidth=1.3, label='Azar')
axes[0].set_xlabel('1 − Especificidad', fontsize=12)
axes[0].set_ylabel('Sensibilidad', fontsize=12)
axes[0].set_title('A. Curva ROC', fontsize=13, fontweight='bold', loc='left')
axes[0].legend(loc='lower right', fontsize=10, frameon=True, edgecolor='#999')
axes[0].grid(alpha=0.25, linestyle=':')
axes[0].set_aspect('equal'); axes[0].set_xlim(0,1); axes[0].set_ylim(0,1.02)

for n, c, ls, mk in estilos:
    pre, rec, _ = precision_recall_curve(all_y[n], all_p[n])
    ap = average_precision_score(all_y[n], all_p[n])
    axes[1].plot(rec, pre, color=c, linewidth=2.4, linestyle=ls,
                  label=f"{n} (AUPRC = {ap:.3f})", alpha=0.95)
axes[1].axhline(y.mean(), color='#444', linestyle=':', linewidth=1.5,
                label=f'Prevalencia ({y.mean()*100:.1f} %)')
axes[1].set_xlabel('Sensibilidad (Recall)', fontsize=12)
axes[1].set_ylabel('Valor predictivo positivo', fontsize=12)
axes[1].set_title('B. Curva Precisión–Recall', fontsize=13, fontweight='bold', loc='left')
axes[1].legend(loc='upper right', fontsize=10, frameon=True, edgecolor='#999')
axes[1].grid(alpha=0.25, linestyle=':')
axes[1].set_xlim(0,1); axes[1].set_ylim(0, 0.6)
plt.tight_layout()
plt.savefig('figura4_ROC_PR.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ figura4_ROC_PR.png")


## Paso 14. Matriz de confusión (Random Forest)

In [ ]:
mejor = 'Random Forest'
y_real = np.array(all_y[mejor])
y_pred = (np.array(all_p[mejor]) >= 0.5).astype(int)
cm = confusion_matrix(y_real, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No MME severa', 'MME severa'],
            yticklabels=['No MME severa', 'MME severa'],
            ax=ax, cbar=False, annot_kws={'fontsize': 14, 'fontweight': 'bold'})
ax.set_xlabel('Predicción del modelo', fontsize=12)
ax.set_ylabel('Realidad observada',    fontsize=12)
ax.set_title(f'Matriz de confusión — {mejor}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('matriz_confusion_RF.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\n=== REPORTE — {mejor} ===")
print(classification_report(y_real, y_pred, target_names=['No MME severa','MME severa']))


## Paso 15. Descargar las figuras

In [ ]:
import zipfile, os
archivos = ['figura2_AUC_comparacion.png','figura3_importancia.png',
            'figura4_ROC_PR.png','figura5_forest_OR.png','matriz_confusion_RF.png']
with zipfile.ZipFile('figuras_resultados.zip', 'w') as zf:
    for f in archivos:
        if os.path.exists(f): zf.write(f)
        else: print(f"⚠️  No se encontró {f}")
files.download('figuras_resultados.zip')
print("✓ figuras_resultados.zip descargado.")


## Conclusión

| Resultado | Esperado |
|---|---|
| Cohorte MME | 5 706 |
| Prevalencia MME severa | ≈ 4,5 % |
| AUC Random Forest | ≈ 0,73 (IC 0,69–0,77) |
| Variable más informativa | Tipo de terminación de la gestación |
| OR pertenencia indígena | 1,72 (IC 1,26–2,34) |
| OR controles prenatales insuficientes | 1,48 (IC 1,12–1,94) |

Si los valores coinciden, la replicación fue exitosa.

### Documentos de transparencia adjuntos
- `checklist_RECORD_diligenciado.docx`
- `checklist_CAIR_diligenciado.docx` (Olczak J, *et al.* Acta Orthop. 2021;92(5):513-525)
- Manuscrito AG-0.9-2026

---
*Cuaderno v2 — Revista Facultad Nacional de Salud Pública (UdeA). Guías: STROBE, RECORD, TRIPOD-AI, CAIR, SRQR.*
